# Phase Kickback — Amazon Braket

**Phase kickback** occurs when a controlled oracle acts on an eigenstate
of the oracle: the eigenvalue phase is "kicked back" onto the control
qubit.  This is the core mechanism behind Deutsch-Jozsa, Grover, and
phase estimation.

In [ ]:
import json

from braket.circuit import Circuit
from braket.devices import LocalSimulator

In [ ]:
device = LocalSimulator()

## Basic kickback with CNOT oracle

Oracle: `CNOT(q0, q1)` computes $f(x) = x$ into the ancilla.
Ancilla starts in $|{-}\rangle = \frac{1}{\sqrt{2}}(|0\rangle - |1\rangle)$.

- Input $|0\rangle$: $|0\rangle|{-}\rangle \to |0\rangle|{-}\rangle$ (no change)
- Input $|1\rangle$: $|1\rangle|{-}\rangle \to -|1\rangle|{-}\rangle$ (phase kickback)

In [ ]:
for inp in (0, 1):
    circuit = Circuit()
    if inp:
        circuit.x(0)
    circuit.h(0)
    circuit.x(1)
    circuit.h(1)
    circuit.cnot(0, 1)

    result = device.run(circuit, shots=0).result()
    amps = result.result_types[0].value
    probs = result.result_types[1].value

    print(f"input |{inp}>: amplitudes = {amps}")
    print(f"         probabilities = {json.dumps({k: round(v, 6) for k, v in probs.items()})}")
    if inp == 0:
        print("  |0>|-> unchanged — no phase kickback.")
    else:
        print("  -|1>|-> — phase -1 kicked back onto q0.")
    print()

## Superposition input

With the input in superposition $\frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$,
kickback produces:
$$\frac{1}{\sqrt{2}}(|0\rangle - |1\rangle)|{-}\rangle$$

Applying $H$ to qubit 0 collapses it to $|1\rangle$.

In [ ]:
circuit = Circuit()
circuit.h(0)
circuit.x(1)
circuit.h(1)
circuit.cnot(0, 1)
circuit.h(0)

result = device.run(circuit, shots=0).result()
amps = result.result_types[0].value
probs = result.result_types[1].value

print(f"amplitudes: {amps}")
print(f"probabilities: {json.dumps({k: round(v, 6) for k, v in probs.items()})}")
print("After H-uncompute, qubit 0 is |1> — the balanced function's")
print("output was kicked back and revealed by the final Hadamard.")

## Phase oracle kickback with $|{+}\rangle$ input

CZ phase oracle on $|{+}\rangle$ input with $|{-}\rangle$ ancilla
gives the cleanest demonstration.

In [ ]:
circuit = Circuit()
circuit.h(0)
circuit.x(1)
circuit.h(1)
circuit.cz(0, 1)

result = device.run(circuit, shots=0).result()
amps = result.result_types[0].value
probs = result.result_types[1].value

print(f"amplitudes: {amps}")
print(f"probabilities: {json.dumps({k: round(v, 6) for k, v in probs.items()})}")
print("The CZ phase oracle applied to |+>|-> gives |->|->.")
print("The phase was kicked back from the ancilla to the input.")